# 1. Load Dataset

In this section, we load the Home Credit application dataset and create a working copy for feature engineering.

We keep the original dataset unchanged so that we can always return to the raw data if needed.

In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("application_train.csv")

# Create working copy
data = df.copy()

print("Shape:", data.shape)

data.head()

Shape: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [2]:
# Missing value summary

missing_df = pd.DataFrame({
    "Missing_Count": data.isnull().sum(),
    "Missing_Percentage": (data.isnull().sum() / len(data)) * 100
})

missing_df = missing_df[missing_df["Missing_Count"] > 0]

missing_df = missing_df.sort_values(
    by="Missing_Percentage",
    ascending=False
)

missing_df.head(20)

,Missing_Count,Missing_Percentage
COMMONAREA_MEDI,214865,69.872297
COMMONAREA_MODE,214865,69.872297
COMMONAREA_AVG,214865,69.872297
NONLIVINGAPARTMENTS_MODE,213514,69.432963
NONLIVINGAPARTMENTS_MEDI,213514,69.432963
NONLIVINGAPARTMENTS_AVG,213514,69.432963
FONDKAPREMONT_MODE,210295,68.386172
LIVINGAPARTMENTS_AVG,210199,68.354953
LIVINGAPARTMENTS_MEDI,210199,68.354953
LIVINGAPARTMENTS_MODE,210199,68.354953


In [3]:
# Find features with very high missing values
missing_percentage = data.isnull().mean() * 100

high_missing_cols = missing_percentage[
    missing_percentage > 70
].index.tolist()

print(f"Columns with >70% missing values: {len(high_missing_cols)}")
high_missing_cols

Columns with >70% missing values: 0


[]

In [4]:
# Columns with more than 70% missing values
high_missing_cols = missing_df[
    missing_df["Missing_Percentage"] > 70
].index.tolist()

print(f"Columns to drop: {len(high_missing_cols)}")

# Drop them
data = data.drop(columns=high_missing_cols)

print("New shape:", data.shape)

Columns to drop: 0
New shape: (307511, 122)


In [5]:
# Select numerical columns
num_cols = data.select_dtypes(include=["int64", "float64"]).columns

# Fill missing numerical values with median
for col in num_cols:
    data[col] = data[col].fillna(data[col].median())

# Check remaining missing values in numerical columns
data[num_cols].isnull().sum().sum()

np.int64(0)

In [6]:
data[num_cols].isnull().sum().sum()

np.int64(0)

In [7]:
# Select categorical columns
cat_cols = data.select_dtypes(include=["object"]).columns

# Fill missing categorical values with 'Unknown'
for col in cat_cols:
    data[col] = data[col].fillna("Unknown")

# Check remaining missing values in categorical columns
data[cat_cols].isnull().sum().sum()

np.int64(0)

In [8]:
# Convert age from negative days to years
data["AGE"] = (-data["DAYS_BIRTH"]) / 365

# Create age groups
data["AGE_GROUP"] = pd.cut(
    data["AGE"],
    bins=[0, 25, 35, 45, 55, 100],
    labels=["<25", "25-35", "35-45", "45-55", "55+"]
)

data[["AGE", "AGE_GROUP"]].head()

C:\Users\adyaa\AppData\Local\Temp\ipykernel_26336\3763202520.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["AGE"] = (-data["DAYS_BIRTH"]) / 365
C:\Users\adyaa\AppData\Local\Temp\ipykernel_26336\3763202520.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["AGE_GROUP"] = pd.cut(


,AGE,AGE_GROUP
0,25.920548,25-35
1,45.931507,45-55
2,52.180822,45-55
3,52.068493,45-55
4,54.608219,45-55


In [9]:
data["CREDIT_INCOME_RATIO"] = (
    data["AMT_CREDIT"] / data["AMT_INCOME_TOTAL"]
)

data["CREDIT_INCOME_RATIO"].describe()

C:\Users\adyaa\AppData\Local\Temp\ipykernel_26336\1971822871.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["CREDIT_INCOME_RATIO"] = (


count    307511.000000
mean          3.957570
std           2.689728
min           0.004808
25%           2.018667
50%           3.265067
75%           5.159880
max          84.736842
Name: CREDIT_INCOME_RATIO, dtype: float64

In [10]:
data["ANNUITY_INCOME_RATIO"] = (
    data["AMT_ANNUITY"] / data["AMT_INCOME_TOTAL"]
)

data["ANNUITY_INCOME_RATIO"].describe()

C:\Users\adyaa\AppData\Local\Temp\ipykernel_26336\4010737204.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["ANNUITY_INCOME_RATIO"] = (


count    307511.000000
mean          0.180929
std           0.094573
min           0.000224
25%           0.114782
50%           0.162833
75%           0.229067
max           1.875965
Name: ANNUITY_INCOME_RATIO, dtype: float64

In [11]:
data["CREDIT_ANNUITY_RATIO"] = (
    data["AMT_CREDIT"] / data["AMT_ANNUITY"]
)

data["CREDIT_ANNUITY_RATIO"].describe()

C:\Users\adyaa\AppData\Local\Temp\ipykernel_26336\4190853032.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["CREDIT_ANNUITY_RATIO"] = (


count    307511.000000
mean         21.612366
std           7.824164
min           6.324539
25%          15.614473
50%          20.000000
75%          27.099985
max          59.560354
Name: CREDIT_ANNUITY_RATIO, dtype: float64

In [12]:
data["INCOME_PER_FAMILY_MEMBER"] = (
    data["AMT_INCOME_TOTAL"] / data["CNT_FAM_MEMBERS"]
)

data["INCOME_PER_FAMILY_MEMBER"].describe()

C:\Users\adyaa\AppData\Local\Temp\ipykernel_26336\398621420.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["INCOME_PER_FAMILY_MEMBER"] = (


count    3.075110e+05
mean     9.310634e+04
std      1.013733e+05
min      2.812500e+03
25%      4.725000e+04
50%      7.500000e+04
75%      1.125000e+05
max      3.900000e+07
Name: INCOME_PER_FAMILY_MEMBER, dtype: float64

In [13]:
data["INCOME_PER_CHILD"] = (
    data["AMT_INCOME_TOTAL"] / (data["CNT_CHILDREN"] + 1)
)

data["INCOME_PER_CHILD"].describe()

C:\Users\adyaa\AppData\Local\Temp\ipykernel_26336\2858085504.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["INCOME_PER_CHILD"] = (


count    3.075110e+05
mean     1.395079e+05
std      1.458110e+05
min      3.000000e+03
25%      7.875000e+04
50%      1.170000e+05
75%      1.800000e+05
max      5.850000e+07
Name: INCOME_PER_CHILD, dtype: float64

In [14]:
data["CHILDREN_RATIO"] = (
    data["CNT_CHILDREN"] / data["CNT_FAM_MEMBERS"]
)

data["CHILDREN_RATIO"].describe()

C:\Users\adyaa\AppData\Local\Temp\ipykernel_26336\4263873308.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["CHILDREN_RATIO"] = (


count    307511.000000
mean          0.125502
std           0.199578
min           0.000000
25%           0.000000
50%           0.000000
75%           0.333333
max           0.950000
Name: CHILDREN_RATIO, dtype: float64

In [15]:
data["EXT_SOURCE_MEAN"] = data[
    ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
].mean(axis=1)

data["EXT_SOURCE_MEAN"].describe()

C:\Users\adyaa\AppData\Local\Temp\ipykernel_26336\3168260113.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["EXT_SOURCE_MEAN"] = data[


count    307511.000000
mean          0.511503
std           0.108104
min           0.023705
25%           0.441604
50%           0.521564
75%           0.587715
max           0.853417
Name: EXT_SOURCE_MEAN, dtype: float64

In [16]:
data["EXT_SOURCE_STD"] = data[
    ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
].std(axis=1)

data["EXT_SOURCE_STD"].describe()

C:\Users\adyaa\AppData\Local\Temp\ipykernel_26336\362723397.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["EXT_SOURCE_STD"] = data[


count    307511.000000
mean          0.141598
std           0.075224
min           0.000261
25%           0.085834
50%           0.130825
75%           0.189184
max           0.486962
Name: EXT_SOURCE_STD, dtype: float64

In [17]:
data["EXT_SOURCE_MIN"] = data[
    ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
].min(axis=1)

data["EXT_SOURCE_MIN"].describe()

C:\Users\adyaa\AppData\Local\Temp\ipykernel_26336\4261277536.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["EXT_SOURCE_MIN"] = data[


count    3.075110e+05
mean     3.720009e-01
std      1.551778e-01
min      8.173617e-08
25%      2.539628e-01
50%      4.034627e-01
75%      5.059979e-01
max      8.068084e-01
Name: EXT_SOURCE_MIN, dtype: float64

In [18]:
data = data.copy()

In [19]:
data["YEARS_REGISTRATION"] = (
    -data["DAYS_REGISTRATION"] / 365
)

data["YEARS_REGISTRATION"].describe()

count    307511.000000
mean         13.660604
std           9.651743
min          -0.000000
25%           5.506849
50%          12.339726
75%          20.491781
max          67.594521
Name: YEARS_REGISTRATION, dtype: float64

In [20]:
data["YEARS_ID_PUBLISH"] = (
    -data["DAYS_ID_PUBLISH"] / 365
)

data["YEARS_ID_PUBLISH"].describe()

count    307511.000000
mean          8.203294
std           4.135481
min           0.000000
25%           4.712329
50%           8.915068
75%          11.778082
max          19.717808
Name: YEARS_ID_PUBLISH, dtype: float64

In [21]:
data["GOODS_CREDIT_RATIO"] = (
    data["AMT_GOODS_PRICE"] / data["AMT_CREDIT"]
)

data["GOODS_CREDIT_RATIO"].describe()

count    307511.000000
mean          0.901702
std           0.104845
min           0.166667
25%           0.834725
50%           0.893815
75%           1.000000
max           6.666667
Name: GOODS_CREDIT_RATIO, dtype: float64

In [22]:
contact_cols = [
    "FLAG_MOBIL",
    "FLAG_EMP_PHONE",
    "FLAG_WORK_PHONE",
    "FLAG_PHONE",
    "FLAG_EMAIL"
]

data["CONTACT_COUNT"] = data[contact_cols].sum(axis=1)

data["CONTACT_COUNT"].value_counts().sort_index()

CONTACT_COUNT
1     37989
2    161225
3     70803
4     35502
5      1992
Name: count, dtype: int64

In [23]:
data["OWNS_CAR"] = (data["FLAG_OWN_CAR"] == "Y").astype(int)

data["OWNS_REALTY"] = (data["FLAG_OWN_REALTY"] == "Y").astype(int)

data[["OWNS_CAR", "OWNS_REALTY"]].head()

,OWNS_CAR,OWNS_REALTY
0,0,1
1,0,0
2,1,1
3,0,1
4,0,1


In [24]:
print(data.shape)

(307511, 139)


In [25]:
income_lower = data["AMT_INCOME_TOTAL"].quantile(0.01)
income_upper = data["AMT_INCOME_TOTAL"].quantile(0.99)

data["AMT_INCOME_TOTAL"] = data["AMT_INCOME_TOTAL"].clip(
    lower=income_lower,
    upper=income_upper
)

data["AMT_INCOME_TOTAL"].describe()

count    307511.000000
mean     166067.479607
std       83000.171936
min       45000.000000
25%      112500.000000
50%      147150.000000
75%      202500.000000
max      472500.000000
Name: AMT_INCOME_TOTAL, dtype: float64

In [26]:
credit_lower = data["AMT_CREDIT"].quantile(0.01)
credit_upper = data["AMT_CREDIT"].quantile(0.99)

data["AMT_CREDIT"] = data["AMT_CREDIT"].clip(
    lower=credit_lower,
    upper=credit_upper
)

data["AMT_CREDIT"].describe()

count    3.075110e+05
mean     5.963060e+05
std      3.913075e+05
min      7.641000e+04
25%      2.700000e+05
50%      5.135310e+05
75%      8.086500e+05
max      1.854000e+06
Name: AMT_CREDIT, dtype: float64

In [27]:
annuity_lower = data["AMT_ANNUITY"].quantile(0.01)
annuity_upper = data["AMT_ANNUITY"].quantile(0.99)

data["AMT_ANNUITY"] = data["AMT_ANNUITY"].clip(
    lower=annuity_lower,
    upper=annuity_upper
)

data["AMT_ANNUITY"].describe()

count    307511.000000
mean      26945.208570
std       13654.548817
min        6183.000000
25%       16524.000000
50%       24903.000000
75%       34596.000000
max       70006.500000
Name: AMT_ANNUITY, dtype: float64

In [28]:
annuity_lower = data["AMT_ANNUITY"].quantile(0.01)
annuity_upper = data["AMT_ANNUITY"].quantile(0.99)

data["AMT_ANNUITY"] = data["AMT_ANNUITY"].clip(
    lower=annuity_lower,
    upper=annuity_upper
)

data["AMT_ANNUITY"].describe()

count    307511.000000
mean      26945.208570
std       13654.548817
min        6183.000000
25%       16524.000000
50%       24903.000000
75%       34596.000000
max       70006.500000
Name: AMT_ANNUITY, dtype: float64

In [29]:
cat_cols = data.select_dtypes(include=["object", "category"]).columns

print("Number of categorical columns:", len(cat_cols))

data[cat_cols].nunique().sort_values()

Number of categorical columns: 17


NAME_CONTRACT_TYPE             2
FLAG_OWN_CAR                   2
FLAG_OWN_REALTY                2
CODE_GENDER                    3
EMERGENCYSTATE_MODE            3
HOUSETYPE_MODE                 4
NAME_EDUCATION_TYPE            5
FONDKAPREMONT_MODE             5
AGE_GROUP                      5
NAME_FAMILY_STATUS             6
NAME_HOUSING_TYPE              6
WEEKDAY_APPR_PROCESS_START     7
NAME_TYPE_SUITE                8
WALLSMATERIAL_MODE             8
NAME_INCOME_TYPE               8
OCCUPATION_TYPE               19
ORGANIZATION_TYPE             58
dtype: int64

In [30]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

binary_cols = []

for col in cat_cols:
    if data[col].nunique() == 2:
        binary_cols.append(col)
        data[col] = le.fit_transform(data[col])

print("Binary columns encoded:", len(binary_cols))

Binary columns encoded: 3


In [31]:
data = pd.get_dummies(
    data,
    columns=[
        col for col in cat_cols
        if col not in binary_cols
    ],
    drop_first=True
)

print(data.shape)

(307511, 256)


In [32]:
print("Shape:", data.shape)

data.head()

Shape: (307511, 256)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,WALLSMATERIAL_MODE_Panel,"WALLSMATERIAL_MODE_Stone, brick",WALLSMATERIAL_MODE_Unknown,WALLSMATERIAL_MODE_Wooden,EMERGENCYSTATE_MODE_Unknown,EMERGENCYSTATE_MODE_Yes,AGE_GROUP_25-35,AGE_GROUP_35-45,AGE_GROUP_45-55,AGE_GROUP_55+
0,100002,1,0,0,1,0,202500.0,406597.5,24700.5,351000.0,...,False,True,False,False,False,False,True,False,False,False
1,100003,0,0,0,0,0,270000.0,1293502.5,35698.5,1129500.0,...,False,False,False,False,False,False,False,False,True,False
2,100004,0,1,1,1,0,67500.0,135000.0,6750.0,135000.0,...,False,False,True,False,True,False,False,False,True,False
3,100006,0,0,0,1,0,135000.0,312682.5,29686.5,297000.0,...,False,False,True,False,True,False,False,False,True,False
4,100007,0,0,0,1,0,121500.0,513000.0,21865.5,513000.0,...,False,False,True,False,True,False,False,False,True,False


In [33]:
constant_cols = [
    col for col in data.columns
    if data[col].nunique() <= 1
]

print("Constant columns:", len(constant_cols))

data.drop(columns=constant_cols, inplace=True)

print(data.shape)

Constant columns: 0
(307511, 256)


In [34]:
corr_matrix = data.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_cols = [
    column
    for column in upper.columns
    if any(upper[column] > 0.90)
]

print("Highly correlated columns:", len(high_corr_cols))

Highly correlated columns: 44


In [35]:
high_corr_cols[:20]

['AMT_GOODS_PRICE',
 'FLAG_EMP_PHONE',
 'REGION_RATING_CLIENT_W_CITY',
 'APARTMENTS_MODE',
 'BASEMENTAREA_MODE',
 'YEARS_BEGINEXPLUATATION_MODE',
 'YEARS_BUILD_MODE',
 'COMMONAREA_MODE',
 'ELEVATORS_MODE',
 'ENTRANCES_MODE',
 'FLOORSMAX_MODE',
 'FLOORSMIN_MODE',
 'LANDAREA_MODE',
 'LIVINGAPARTMENTS_MODE',
 'LIVINGAREA_MODE',
 'NONLIVINGAPARTMENTS_MODE',
 'NONLIVINGAREA_MODE',
 'APARTMENTS_MEDI',
 'BASEMENTAREA_MEDI',
 'YEARS_BEGINEXPLUATATION_MEDI']

In [36]:
data = data.drop(columns=high_corr_cols)

print(data.shape)

(307511, 212)


In [37]:
data.to_csv(
    "home_credit_processed.csv",
    index=False
)

print("Dataset saved successfully")

Dataset saved successfully


In [38]:
print("Final shape:", data.shape)
print("Missing values:", data.isnull().sum().sum())

Final shape: (307511, 212)
Missing values: 0
